# Learning and Adaptation

The Learning and Adaptation pattern enables agents to iteratively improve their outputs through an evaluate → select → mutate → repeat cycle. Inspired by evolutionary algorithms and systems like SICA and OpenEvolve, the agent uses an LLM to generate variants of a program, scores them, keeps the best, and repeats until a fitness threshold is reached or iterations are exhausted.

## Implementation with Flyte v2

This notebook reimplements the OpenEvolve pattern from Chapter 9 using **Flyte v2 primitives + Anthropic API** — no `openevolve` library required.

#### OpenEvolve vs Flyte v2 — Key Differences

| Aspect | OpenEvolve | Flyte v2 |
|--------|-----------|----------|
| **Evolution loop** | `OpenEvolve(initial_program_path, evaluation_file, config_path)` + `await evolve.run(iterations=N)` | Explicit async loop inside `@env.task` |
| **Code sampling** | LLM ensemble via config YAML | Direct `AsyncAnthropic` call with mutation prompt |
| **Evaluation** | Separate `evaluator.py` script executed as subprocess | Inline `exec()` sandbox with captured metrics |
| **State** | File-based (program `.py` files on disk) | Typed `ProgramVariant` dataclass — serializable, inspectable |
| **Distributed eval** | Built-in evaluator pool | `asyncio.gather` over variants |
| **Progress** | Console logs | `flyte.report` HTML tab updated each generation |
| **Secrets** | Config YAML / `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' openai

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret OPENAI_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta
from typing import Optional

from openai import AsyncOpenAI, OpenAI
import flyte
import flyte.report

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="evo-agent", python_version=(3, 12))
    .with_pip_packages("openai>=1.0.0")
)

evo_env = flyte.TaskEnvironment(
    name="evo_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY"),
    ],
)

### 4. Define data models

OpenEvolve stores candidate programs as files and tracks metrics in a `Program` database. In Flyte v2, typed dataclasses replace file-based state — every generation is a serializable `ProgramVariant` that is visible as structured output in the UI.

In [ ]:
@dataclass
class ProgramVariant:
    """A single candidate program and its evaluation score."""
    code: str
    score: float = 0.0
    generation: int = 0
    mutation_notes: str = ""


@dataclass
class EvoResult:
    """Final output of the evolutionary coding agent."""
    task_description: str
    best_code: str
    best_score: float
    generations: int
    total_variants_tried: int
    converged: bool

### 5. Define the LLM-based mutation and evaluation helpers

OpenEvolve uses a config YAML to define LLM ensembles for sampling and a separate `evaluator.py` script for scoring. In Flyte v2:
- `_mutate()` replaces the sampler — asks the LLM to improve the code with a specific critique
- `_evaluate()` replaces the evaluator script — runs the code in a sandbox and measures correctness

Both are decorated with `@flyte.trace` for per-call checkpointing.

In [ ]:
MUTATION_SYSTEM = """\
You are an expert Python programmer. Given a function and feedback, produce an improved version.

Rules:
- Return ONLY the complete improved Python function, no explanation.
- Keep the same function signature.
- Make exactly one focused improvement per mutation.
- The function must be self-contained (no imports outside the function body)."""


@flyte.trace
async def _mutate(
    current_code: str,
    task_description: str,
    critique: str,
) -> ProgramVariant:
    """Ask the LLM to produce an improved variant. Replaces OpenEvolve's LLM sampler."""
    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=1024,
        messages=[{"role": "system", "content": MUTATION_SYSTEM}] + [{
            "role": "user",
            "content": (
                f"Task: {task_description}\n\n"
                f"Current code:\n```python\n{current_code}\n```\n\n"
                f"Critique / improvement direction: {critique}\n\n"
                "Produce the improved function:"
            ),
        }],
    )
    raw = response.choices[0].message.content.strip()
    # Strip markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return ProgramVariant(code=raw, mutation_notes=critique)


@flyte.trace
async def _evaluate(
    variant: ProgramVariant,
    test_cases: list[tuple],
    fn_name: str,
) -> ProgramVariant:
    """Score a variant by running test cases. Replaces OpenEvolve's evaluator.py script."""
    namespace: dict = {}
    try:
        exec(compile(variant.code, "<variant>", "exec"), namespace)  # noqa: S102
    except Exception as e:
        return ProgramVariant(
            code=variant.code,
            score=0.0,
            generation=variant.generation,
            mutation_notes=f"compile error: {e}",
        )

    fn = namespace.get(fn_name)
    if not callable(fn):
        return ProgramVariant(code=variant.code, score=0.0, generation=variant.generation,
                              mutation_notes="function not found after exec")

    passed = 0
    for inputs, expected in test_cases:
        try:
            result = fn(*inputs) if isinstance(inputs, tuple) else fn(inputs)
            if result == expected:
                passed += 1
        except Exception:
            pass

    score = passed / len(test_cases) if test_cases else 0.0
    return ProgramVariant(
        code=variant.code,
        score=score,
        generation=variant.generation,
        mutation_notes=variant.mutation_notes,
    )


@flyte.trace
async def _critique(
    variant: ProgramVariant,
    task_description: str,
    test_cases: list[tuple],
    fn_name: str,
) -> str:
    """Ask the LLM to critique the current best program and suggest a mutation direction."""
    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
    failed = [
        f"  {inp} → expected {exp}"
        for inp, exp in test_cases
    ]
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=256,
        messages=[{"role": "system", "content": "You are a code reviewer. Suggest ONE specific improvement for a Python function."}] + [{
            "role": "user",
            "content": (
                f"Task: {task_description}\n"
                f"Current score: {variant.score:.2f}\n"
                f"Code:\n```python\n{variant.code}\n```\n"
                f"Test cases (input → expected):\n" + "\n".join(failed) + "\n\n"
                "In one sentence, what should the next mutation focus on?"
            ),
        }],
    )
    return response.choices[0].message.content.strip()

### 6. Define the evolutionary coding agent task

OpenEvolve's `await evolve.run(iterations=1000)` hides a generate → evaluate → select loop inside the library. In Flyte v2, this loop is explicit Python — the same logic, fully transparent:

1. **Mutate** the current best program with LLM guidance
2. **Evaluate** the variant against test cases
3. **Select** the better of old vs. new (elitist selection)
4. Update the live report and repeat

`@flyte.trace` on each helper means the task resumes from the last checkpoint on pod failure.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@evo_env.task(
    retries=2,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def evolve_program(
    task_description: str,
    initial_code: str,
    test_cases: list[list],   # list of [inputs, expected]; inputs is list for multi-arg
    fn_name: str,
    max_generations: int = 10,
    target_score: float = 1.0,
) -> EvoResult:
    """
    Evolutionary coding agent: iteratively improve a Python function.

    Replaces OpenEvolve's:
      evolve = OpenEvolve(initial_program_path, evaluation_file, config_path)
      best_program = await evolve.run(iterations=1000)

    The generate → evaluate → select loop is the same logic, now explicit.
    """
    # Convert nested lists to tuples for test_cases
    tc: list[tuple] = [(tuple(inp) if isinstance(inp, list) else inp, exp)
                       for inp, exp in test_cases]

    best = await _evaluate(
        ProgramVariant(code=initial_code, generation=0),
        tc, fn_name,
    )
    total_tried = 1
    report_rows: list[str] = []

    for gen in range(1, max_generations + 1):
        # Critique current best → get mutation direction
        critique = await _critique(best, task_description, tc, fn_name)

        # Generate a mutant
        mutant = await _mutate(best.code, task_description, critique)
        mutant = ProgramVariant(
            code=mutant.code,
            generation=gen,
            mutation_notes=mutant.mutation_notes,
        )

        # Evaluate the mutant
        mutant = await _evaluate(mutant, tc, fn_name)
        total_tried += 1

        # Elitist selection: keep the better program
        improved = mutant.score > best.score
        if improved:
            best = mutant

        color = "green" if improved else "gray"
        report_rows.append(
            f"<tr><td>{gen}</td>"
            f"<td style='color:{color}'>{mutant.score:.2f}</td>"
            f"<td>{best.score:.2f}</td>"
            f"<td>{_html_escape(critique[:80])}...</td></tr>"
        )

        await flyte.report.replace.aio(
            "<!DOCTYPE html><html><body>"
            f"<h2>Evolving: {_html_escape(task_description[:60])}</h2>"
            f"<p>Generation {gen} / {max_generations} | Best score: <strong>{best.score:.2f}</strong></p>"
            "<table border='1' cellpadding='4'>"
            "<tr><th>Gen</th><th>Mutant</th><th>Best</th><th>Critique</th></tr>"
            + "".join(report_rows)
            + "</table>"
            f"<h3>Current best code</h3><pre>{_html_escape(best.code)}</pre>"
            "</body></html>"
        )
        await flyte.report.flush.aio()

        if best.score >= target_score:
            break

    return EvoResult(
        task_description=task_description,
        best_code=best.code,
        best_score=best.score,
        generations=gen,
        total_variants_tried=total_tried,
        converged=best.score >= target_score,
    )

### 7. Run locally

In [ ]:
# Start with a broken implementation — the agent must fix it
INITIAL_CODE = """
def is_palindrome(s: str) -> bool:
    # placeholder — always returns False
    return False
""".strip()

TEST_CASES = [
    [["racecar"], True],
    [["hello"], False],
    [["level"], True],
    [["world"], False],
    [["madam"], True],
]

run = flyte.run(
    evolve_program,
    task_description="A function is_palindrome(s) that returns True if s reads the same forwards and backwards.",
    initial_code=INITIAL_CODE,
    test_cases=TEST_CASES,
    fn_name="is_palindrome",
    max_generations=8,
    target_score=1.0,
)
run.wait()
result: EvoResult = run.outputs()[0]

print(f"Converged: {result.converged}")
print(f"Best score: {result.best_score:.2f} after {result.generations} generations ({result.total_variants_tried} variants)")
print(f"\nBest code:\n{result.best_code}")

### Running remotely

The `report=True` flag enables a live HTML tab in the Flyte UI — watch each generation's score and critique update in real time as the agent evolves the program. The `EvoResult` dataclass makes the final best code and convergence metrics inspectable without parsing logs.

In [ ]:
run = flyte.run(
    evolve_program,
    task_description="A function binary_search(arr, target) returning the index of target in sorted arr, or -1 if not found.",
    initial_code="def binary_search(arr, target):\n    return -1  # placeholder",
    test_cases=[
        [[[1, 3, 5, 7, 9], 5], 2],
        [[[1, 3, 5, 7, 9], 1], 0],
        [[[1, 3, 5, 7, 9], 9], 4],
        [[[1, 3, 5, 7, 9], 4], -1],
    ],
    fn_name="binary_search",
    max_generations=12,
    target_score=1.0,
)
run.wait()
result = run.outputs()[0]
print(f"Score: {result.best_score:.2f}, Generations: {result.generations}")
print(result.best_code)

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_evo_agent = flyte.TaskEnvironment(
    name="evo_agent_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)